# Trabalho Final Prático — Inteligência Artificial
## Sistema Inteligente de Análise de Risco de Crédito
### Lógica Fuzzy + Machine Learning (Integração)

**Disciplina:** Inteligência Artificial — Prof. William Malvezzi
**Instituição:** Universidade Católica de Brasília (UCB)
**Grupo:** Grupo 4
**Tema escolhido:** Tema 4 — Concessão de Crédito

---

### Resumo da solução
Construímos um *"analista de crédito digital"* que decide se um solicitante é de **risco baixo, médio ou alto**, combinando **duas abordagens da disciplina**:

1. **🧠 Machine Learning (Árvore de Decisão):** aprende, a partir de 1.000 pedidos de crédito reais, o padrão de quem se torna inadimplente. Produz uma **probabilidade de inadimplência** (0–100%).
2. **🧩 Lógica Fuzzy:** recebe essa probabilidade — junto com o *comprometimento da renda* e o *histórico de crédito* — e aplica **regras SE–ENTÃO** para gerar um **risco final interpretável**.

A articulação entre as duas é por **Integração** (Abordagem B do enunciado): *a saída do ML vira entrada do sistema fuzzy*.

```
Dados do cliente → [ Árvore de Decisão ] → prob. de inadimplência → [ Sistema Fuzzy ] → Risco final (baixo/médio/alto)
```

**Base de dados:** *Statlog (German Credit Data)* — UCI Machine Learning Repository (1.000 registros, base pública).


## Fase 1 — Preparação do Ambiente

A **primeira célula** instala a biblioteca de lógica fuzzy (`scikit-fuzzy`), que não vem por padrão no Colab. A **segunda** importa tudo e fixa uma **semente aleatória** (`random_state=42`) para que **todos os resultados sejam reproduzíveis** — qualquer integrante do grupo (ou o professor) que rodar o notebook obtém exatamente os mesmos números.

> Se, após instalar, o Colab pedir para **reiniciar o ambiente**, reinicie e use *Executar tudo* novamente.


In [ ]:
# Passo 0: instalar o scikit-fuzzy (rode esta celula PRIMEIRO).
# Se o Colab pedir para reiniciar o ambiente apos instalar, reinicie e rode tudo de novo.
!pip install scikit-fuzzy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

# Logica Fuzzy
import skfuzzy as fuzz
from skfuzzy import control as ctrl

import warnings
warnings.filterwarnings('ignore')   # mantem a saida limpa

# Semente: garante reprodutibilidade (mesmos resultados para todo o grupo)
SEMENTE = 42
np.random.seed(SEMENTE)

print("Ambiente pronto! Bibliotecas importadas com sucesso.")


## Fase 2 — Carregar e Entender a Base de Dados

Usamos a versão **completa** do *German Credit Data* (UCI), que tem **20 atributos** por cliente — incluindo as colunas de **histórico de crédito** e **taxa de comprometimento da renda** que o nosso sistema fuzzy vai usar.

O arquivo `german.data` não tem cabeçalho e usa **códigos** (ex.: `A34`), então atribuímos os nomes das colunas manualmente (conforme o dicionário oficial da base).

> **Alvo (variável que queremos prever):** na base original, `1 = bom pagador` e `2 = mau pagador`. Convertemos para `inadimplente` → `0 = pagou` / `1 = não pagou`.


In [ ]:
# Nomes das 20 colunas + a classe (conforme dicionario oficial do UCI)
colunas = ['checking', 'duration', 'credit_history', 'purpose', 'credit_amount',
           'savings', 'employment', 'installment_rate', 'personal_status', 'other_debtors',
           'residence_since', 'property', 'age', 'other_plans', 'housing',
           'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker', 'class']

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

try:
    # A base e separada por espacos (sep=\s+ lida com um ou mais espacos)
    df = pd.read_csv(URL, sep=r'\s+', header=None, names=colunas)
    print("Base carregada direto do UCI.")
except Exception as e:
    # Plano B: pacote oficial do UCI, caso o link direto falhe
    print("Link direto falhou, usando o pacote ucimlrepo...", e)
    !pip install ucimlrepo -q
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=144)
    df = ds.data.features.copy()
    df.columns = colunas[:-1]
    df['class'] = ds.data.targets.values.ravel()

# Cria a variavel-alvo: 1 = inadimplente (mau pagador), 0 = adimplente
df['inadimplente'] = (df['class'] == 2).astype(int)

print("Formato da base:", df.shape, "(linhas, colunas)")
df.head()


In [ ]:
# Visao geral da base
print("===== INFORMACOES GERAIS =====")
df.info()

print("\n===== VALORES FALTANTES POR COLUNA =====")
print(df.isnull().sum().sum(), "valores faltantes no total")

print("\n===== ESTATISTICAS DAS VARIAVEIS NUMERICAS =====")
df.describe()


**Dicionário das principais colunas** (as que mais nos interessam):

| Coluna | Significado |
|---|---|
| `credit_history` | histórico de pagamento (códigos A30–A34) → vira o `score_historico` do fuzzy |
| `installment_rate` | taxa da parcela em % da renda (1–4) → vira o `comprometimento` do fuzzy |
| `credit_amount` | valor do crédito solicitado |
| `duration` | duração do crédito (meses) |
| `age` | idade do solicitante |
| `inadimplente` | **alvo**: 1 = não pagou, 0 = pagou |


## Fase 3 — Análise Exploratória (EDA) e Pré-processamento

Antes de treinar qualquer modelo, precisamos **entender** os dados e **prepará-los**.


In [ ]:
# 1) Balanceamento das classes: quantos pagaram vs nao pagaram?
contagem = df['inadimplente'].value_counts()
print("Adimplentes (0):", contagem[0], "| Inadimplentes (1):", contagem[1])
print(f"Proporcao de inadimplentes: {df['inadimplente'].mean()*100:.1f}%")

plt.figure(figsize=(5,4))
sns.countplot(x='inadimplente', hue='inadimplente', data=df,
              palette=['#2e7d32', '#c62828'], legend=False)
plt.title("Balanceamento das classes")
plt.xticks([0,1], ['Adimplente (0)', 'Inadimplente (1)'])
plt.ylabel("Quantidade de clientes")
plt.show()


In [ ]:
# 2) Distribuicao de algumas variaveis numericas
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col, titulo in zip(axes,
                           ['age', 'credit_amount', 'duration'],
                           ['Idade', 'Valor do credito', 'Duracao (meses)']):
    sns.histplot(df[col], kde=True, ax=ax, color='#1565c0')
    ax.set_title(titulo)
plt.tight_layout()
plt.show()


### Verificação obrigatória: a direção da escala de `installment_rate`

A documentação do UCI **não diz** se um `installment_rate` maior significa parcela maior (mais comprometimento) ou menor. Em vez de chutar, **descobrimos pelos próprios dados**: vemos se a inadimplência *sobe* ou *desce* conforme o valor aumenta. Assim a normalização para `comprometimento` fica correta e justificada.


In [ ]:
# Inadimplencia media para cada valor de installment_rate
print("Inadimplencia media por installment_rate:")
print(df.groupby('installment_rate')['inadimplente'].mean())

# Correlacao com o alvo decide a direcao automaticamente
correlacao = df['installment_rate'].corr(df['inadimplente'])
print(f"\nCorrelacao installment_rate x inadimplencia: {correlacao:.3f}")

if correlacao >= 0:
    # valor alto = mais inadimplencia = mais comprometimento -> normaliza direto
    df['comprometimento'] = (df['installment_rate'] - 1) / 3 * 100
    print("=> Valor ALTO indica MAIOR comprometimento (normalizacao direta).")
else:
    # valor alto = menos comprometimento -> inverte
    df['comprometimento'] = (4 - df['installment_rate']) / 3 * 100
    print("=> Valor ALTO indica MENOR comprometimento (escala invertida).")


### Construindo as variáveis de entrada do fuzzy

- **`comprometimento`** (0–100): acabou de ser criado a partir do `installment_rate`.
- **`score_historico`** (0–100): convertemos o histórico de crédito (códigos A30–A34) em uma nota, por **regra de negócio**.


In [ ]:
# Mapa de negocio: historico de credito -> nota de 0 a 100
# (quanto melhor o historico de pagamento, maior a nota)
mapa_score = {
    'A34': 10,   # conta critica / outros creditos -> pessimo
    'A33': 30,   # atraso no pagamento no passado  -> ruim
    'A30': 50,   # nenhum credito / todos quitados -> neutro
    'A32': 70,   # creditos existentes pagos em dia -> bom
    'A31': 90,   # todos os creditos no banco pagos em dia -> otimo
}
df['score_historico'] = df['credit_history'].map(mapa_score)

print("Distribuicao do score_historico criado:")
print(df['score_historico'].value_counts().sort_index())
df[['credit_history', 'score_historico', 'installment_rate', 'comprometimento']].head()


### Encoding das variáveis categóricas

Vários atributos vêm como **texto/código** (ex.: `A11`, `A34`). O modelo de Machine Learning só entende **números**, então convertemos cada coluna categórica em números (`LabelEncoder`).

Em seguida, separamos a base em **treino (80%)** e **teste (20%)**. O modelo aprende no treino e é avaliado no teste — dados que ele **nunca viu** (senão seria "decorar a prova").


In [ ]:
# Copia para o ML (mantemos df original intacto para o fuzzy)
df_ml = df.copy()

# Colunas que sao texto/codigo precisam virar numero
colunas_categoricas = df_ml.select_dtypes(include='object').columns
print("Colunas categoricas que serao codificadas:", list(colunas_categoricas))

for col in colunas_categoricas:
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col])

# X = atributos de entrada (os 20 originais) ; y = alvo
# Removemos 'class', 'inadimplente' e as colunas derivadas para o fuzzy
features = colunas[:-1]          # os 20 atributos originais
X = df_ml[features]
y = df_ml['inadimplente']

# Dividimos pelos INDICES para conseguir recuperar comprometimento/score depois
idx_treino, idx_teste = train_test_split(
    df.index, test_size=0.2, random_state=SEMENTE, stratify=y)

X_treino, X_teste = X.loc[idx_treino], X.loc[idx_teste]
y_treino, y_teste = y.loc[idx_treino], y.loc[idx_teste]

print(f"\nTreino: {len(X_treino)} clientes | Teste: {len(X_teste)} clientes")


## Fase 4 — Modelo de Machine Learning (Árvore de Decisão)

Escolhemos a **Árvore de Decisão** porque é o algoritmo de **preferência da disciplina** e, principalmente, porque é **interpretável**: dá para enxergar as "perguntas" que ele faz para decidir — o que facilita a comparação com as regras fuzzy.

Limitamos a profundidade (`max_depth=5`) para a árvore não "decorar" o treino (overfitting) e generalizar melhor no teste.


In [ ]:
# Cria e treina a arvore
arvore = DecisionTreeClassifier(max_depth=5, random_state=SEMENTE)
arvore.fit(X_treino, y_treino)

# Previsoes no conjunto de teste
y_prev = arvore.predict(X_teste)

print("Modelo treinado com sucesso!")


### Avaliação do modelo — e o que cada métrica significa

| Métrica | O que mede (em palavras simples) |
|---|---|
| **Acurácia** | % de acertos no total |
| **Matriz de confusão** | mostra *onde* o modelo errou (confundiu bom com mau e vice-versa) |
| **Precisão** | dos que o modelo chamou de inadimplentes, quantos eram mesmo? |
| **Recall** | dos inadimplentes reais, quantos o modelo conseguiu pegar? |
| **F1-score** | equilíbrio entre precisão e recall (média harmônica) |

> No crédito, o **recall dos inadimplentes** é especialmente importante: deixar passar um mau pagador (aprovar quem não vai pagar) costuma custar mais caro que recusar um bom pagador.


In [ ]:
print(f"Acuracia: {accuracy_score(y_teste, y_prev)*100:.1f}%\n")

print("Relatorio de classificacao:")
print(classification_report(y_teste, y_prev,
      target_names=['Adimplente', 'Inadimplente']))

# Matriz de confusao
cm = confusion_matrix(y_teste, y_prev)
ConfusionMatrixDisplay(cm, display_labels=['Adimplente', 'Inadimplente']).plot(cmap='Blues')
plt.title("Matriz de Confusao")
plt.show()


In [ ]:
# Visualizacao da arvore: as 'perguntas' que o modelo aprendeu
plt.figure(figsize=(20,10))
plot_tree(arvore, feature_names=features, class_names=['Adimplente', 'Inadimplente'],
          filled=True, rounded=True, fontsize=8, max_depth=3)  # max_depth=3 so para a figura
plt.title("Arvore de Decisao (3 primeiros niveis, para leitura)")
plt.show()

# Quais variaveis o modelo achou mais importantes?
importancias = pd.Series(arvore.feature_importances_, index=features).sort_values(ascending=False)
print("Top 8 variaveis mais importantes para o modelo:")
print(importancias.head(8))


### A ponte entre os dois cérebros: `predict_proba`

Em vez de só dizer "bom" ou "mau", pedimos ao modelo a **probabilidade de inadimplência** de cada cliente. É esse número (0–100%) que vamos entregar ao sistema fuzzy.


In [ ]:
# Probabilidade de inadimplencia (classe 1) para cada cliente do teste, em %
prob_inadimplencia_teste = arvore.predict_proba(X_teste)[:, 1] * 100

# Mostra alguns exemplos
exemplo = pd.DataFrame({
    'prob_inadimplencia_%': prob_inadimplencia_teste[:5].round(1),
    'inadimplente_real': y_teste.values[:5]
})
print("Exemplo: probabilidade prevista vs realidade")
print(exemplo)


## Fase 5 — Sistema de Inferência Fuzzy

Agora construímos o segundo cérebro. Ele tem:

- **3 variáveis de entrada:** `prob_inadimplencia` (vinda do ML), `comprometimento` e `score_historico`.
- **1 variável de saída:** `risco_credito` (0–100).
- **3 termos linguísticos** por variável (ex.: baixo / médio / alto).
- **Funções de pertinência triangulares** (`trimf`), conforme a preferência do enunciado.

Todas as variáveis usam o universo de **0 a 100** para ficar uniforme.

> **Defuzzificação:** ao final, o resultado fuzzy é convertido de volta para um número (0–100) pelo método do **centroide (centro de gravidade)** — o padrão do `scikit-fuzzy`. Deixamos essa escolha explícita no código.


In [ ]:
# --- Variaveis de ENTRADA (Antecedentes) e SAIDA (Consequente) ---
prob = ctrl.Antecedent(np.arange(0, 101, 1), 'prob_inadimplencia')
comp = ctrl.Antecedent(np.arange(0, 101, 1), 'comprometimento')
hist = ctrl.Antecedent(np.arange(0, 101, 1), 'score_historico')
risco = ctrl.Consequent(np.arange(0, 101, 1), 'risco_credito')

# --- Funcoes de pertinencia triangulares ---
# Probabilidade de inadimplencia
prob['baixa'] = fuzz.trimf(prob.universe, [0, 0, 40])
prob['media'] = fuzz.trimf(prob.universe, [20, 50, 80])
prob['alta']  = fuzz.trimf(prob.universe, [60, 100, 100])

# Comprometimento da renda
comp['baixo'] = fuzz.trimf(comp.universe, [0, 0, 40])
comp['medio'] = fuzz.trimf(comp.universe, [20, 50, 80])
comp['alto']  = fuzz.trimf(comp.universe, [60, 100, 100])

# Score do historico de credito
hist['ruim']    = fuzz.trimf(hist.universe, [0, 0, 40])
hist['regular'] = fuzz.trimf(hist.universe, [20, 50, 80])
hist['bom']     = fuzz.trimf(hist.universe, [60, 100, 100])

# Risco de credito (saida)
risco['baixo'] = fuzz.trimf(risco.universe, [0, 0, 40])
risco['medio'] = fuzz.trimf(risco.universe, [20, 50, 80])
risco['alto']  = fuzz.trimf(risco.universe, [60, 100, 100])

# Defuzzificacao pelo metodo do centroide (centro de gravidade) - deixado explicito
risco.defuzzify_method = 'centroid'

print("Variaveis fuzzy e funcoes de pertinencia definidas.")


In [ ]:
# Visualizando as funcoes de pertinencia
prob.view();  plt.title("Probabilidade de inadimplencia")
comp.view();  plt.title("Comprometimento da renda")
hist.view();  plt.title("Score do historico")
risco.view(); plt.title("Risco de credito (saida)")
plt.show()


### As regras de inferência (SE … ENTÃO)

Construímos **9 regras próprias**, em dois grupos:

- **3 regras-base (cobertura):** ancoram a decisão na probabilidade do ML. Como as três faixas de `prob` cobrem todo o intervalo 0–100, **sempre há pelo menos uma regra ativa** — isso garante que o sistema nunca fica sem resposta.
- **6 regras de refinamento:** ajustam o risco usando o *comprometimento* e o *histórico*, como faria um analista de crédito.

> O enunciado pede no mínimo 6 regras; usamos 9 para tornar o sistema robusto e mais realista.


In [ ]:
# --- Regras-base: ancoradas na probabilidade do ML (garantem cobertura) ---
r1 = ctrl.Rule(prob['alta'],  risco['alto'])
r2 = ctrl.Rule(prob['media'], risco['medio'])
r3 = ctrl.Rule(prob['baixa'], risco['baixo'])

# --- Regras de refinamento: temperam com comprometimento e historico ---
r4 = ctrl.Rule(prob['alta']  & comp['alto'],  risco['alto'])
r5 = ctrl.Rule(prob['alta']  & hist['ruim'],  risco['alto'])
r6 = ctrl.Rule(prob['baixa'] & hist['bom'],   risco['baixo'])
r7 = ctrl.Rule(prob['baixa'] & comp['baixo'], risco['baixo'])
r8 = ctrl.Rule(prob['media'] & hist['regular'], risco['medio'])
r9 = ctrl.Rule(prob['media'] & comp['medio'] & hist['regular'], risco['medio'])

# Monta o sistema de controle fuzzy e o simulador
sistema = ctrl.ControlSystem([r1, r2, r3, r4, r5, r6, r7, r8, r9])
simulador = ctrl.ControlSystemSimulation(sistema)

print("Sistema fuzzy montado com 9 regras.")


## Fase 6 — Integração: ligando o ML no Fuzzy

Esta é a parte central do trabalho. Para **cada cliente do conjunto de teste**:
1. pegamos a **probabilidade de inadimplência** que a Árvore de Decisão calculou;
2. juntamos com o **comprometimento** e o **score do histórico** daquele cliente;
3. passamos os três pela **fuzzificação → inferência → defuzzificação**;
4. obtemos um **risco final** (número 0–100), que rotulamos como **baixo / médio / alto**.


In [ ]:
contador_fallback = {'n': 0}   # conta quantas vezes o fuzzy nao computou (deve ficar 0)

def classificar_risco(p, c, s):
    """Recebe prob. de inadimplencia, comprometimento e score; devolve (valor, rotulo)."""
    try:
        simulador.input['prob_inadimplencia'] = p
        simulador.input['comprometimento']   = c
        simulador.input['score_historico']   = s
        simulador.compute()                       # fuzzificacao + inferencia + defuzzificacao
        valor = simulador.output['risco_credito']
    except Exception as e:
        # rede de seguranca: se nenhuma regra disparar, usa a propria probabilidade do ML
        contador_fallback['n'] += 1
        valor = p

    if valor <= 33:
        rotulo = 'baixo'
    elif valor <= 66:
        rotulo = 'medio'
    else:
        rotulo = 'alto'
    return valor, rotulo

# Aplica a TODOS os clientes do teste
resultados = []
for pos, i in enumerate(idx_teste):
    p = prob_inadimplencia_teste[pos]
    c = df.loc[i, 'comprometimento']
    s = df.loc[i, 'score_historico']
    valor, rotulo = classificar_risco(p, c, s)
    resultados.append({
        'prob_inadimplencia': round(p, 1),
        'comprometimento': round(c, 1),
        'score_historico': s,
        'risco_valor': round(valor, 1),
        'risco_final': rotulo,
        'inadimplente_real': df.loc[i, 'inadimplente']
    })

resultados = pd.DataFrame(resultados, index=idx_teste)
print("Integracao concluida para", len(resultados), "clientes.")
print("Clientes que precisaram de fallback (fuzzy sem regra ativa):", contador_fallback['n'])
resultados.head(10)


In [ ]:
# Como ficou a distribuicao do risco final?
plt.figure(figsize=(6,4))
ordem = ['baixo', 'medio', 'alto']
sns.countplot(x='risco_final', hue='risco_final', data=resultados, order=ordem,
              palette=['#2e7d32', '#f9a825', '#c62828'], legend=False)
plt.title("Distribuicao do risco final (sistema integrado)")
plt.xlabel("Risco de credito"); plt.ylabel("Qtd. de clientes")
plt.show()

# O risco final faz sentido? Cruzamos com a inadimplencia REAL
print("Inadimplencia real media por faixa de risco atribuida:")
print(resultados.groupby('risco_final')['inadimplente_real'].mean().reindex(ordem))
print("\n(Esperado: risco 'alto' deve ter inadimplencia real maior que 'baixo'.)")


### O que a Lógica Fuzzy *acrescenta* ao Machine Learning?

Para mostrar que a integração não é só "reembrulhar" a saída do modelo, comparamos o risco final (fuzzy) com o que seria usar **apenas a probabilidade do ML** dividida nas mesmas faixas. Os casos em que eles **diferem** são exatamente onde o comprometimento da renda e o histórico de crédito *deslocaram* a decisão — o ganho de interpretabilidade da camada fuzzy.


In [ ]:
# Faixa que sairia usando SO a probabilidade do ML
def faixa_direta(p):
    return 'baixo' if p <= 33 else ('medio' if p <= 66 else 'alto')

resultados['risco_so_ML'] = resultados['prob_inadimplencia'].apply(faixa_direta)

mudaram = (resultados['risco_final'] != resultados['risco_so_ML']).sum()
print(f"O sistema fuzzy mudou a classificacao de {mudaram} de {len(resultados)} clientes "
      f"({mudaram/len(resultados)*100:.1f}%) em relacao a usar so a probabilidade do ML.\n")

print("Exemplos onde comprometimento/historico deslocaram o risco:")
diferentes = resultados[resultados['risco_final'] != resultados['risco_so_ML']]
print(diferentes[['prob_inadimplencia', 'comprometimento', 'score_historico',
                  'risco_so_ML', 'risco_final']].head(5))


## Fase 7 — Cenários de Teste e Discussão

Para deixar o funcionamento do sistema fuzzy bem claro, testamos **perfis representativos** de clientes, alimentando o sistema diretamente com valores escolhidos.


In [ ]:
cenarios = [
    ("Cliente otimo",   10, 20, 90),
    ("Cliente mediano", 45, 50, 50),
    ("Cliente ruim",    80, 75, 20),
    ("Cliente ambiguo", 55, 30, 70),
]

print(f"{'Perfil':<18}{'Prob%':>7}{'Compr%':>8}{'Score':>7}{'Risco':>8}   Classificacao")
print("-"*65)
for nome, p, c, s in cenarios:
    valor, rotulo = classificar_risco(p, c, s)
    print(f"{nome:<18}{p:>7}{c:>8}{s:>7}{valor:>8.1f}   {rotulo.upper()}")


In [ ]:
# Grafico da defuzzificacao para um cenario (cliente ruim)
classificar_risco(80, 75, 20)   # recomputa com o ultimo cenario
risco.view(sim=simulador)
plt.title("Defuzzificacao - Cliente ruim (prob=80, compr=75, score=20)")
plt.show()


### Discussão crítica

**Os resultados são coerentes?**
- Sim: o cliente *ótimo* recebe risco **baixo** e o *ruim* recebe risco **alto**. No cruzamento da Fase 6, a faixa de risco **alto** concentra inadimplência real maior que a faixa **baixo** — ou seja, o sistema integrado separa bem os perfis.

**O papel de cada abordagem (Integração):**
- A **Árvore de Decisão** aprende, sozinha e a partir dos dados históricos, padrões que seriam difíceis de escrever à mão.
- A **Lógica Fuzzy** pega esse "instinto" do modelo (a probabilidade) e o traduz em uma decisão **explicável por regras de negócio** — algo que um gerente de crédito entende e justifica.
- Observação honesta: a probabilidade do ML **já considera** comprometimento e histórico internamente; usá-los de novo no fuzzy é **intencional** — é a camada de regras de negócio que *tempera* a decisão do modelo, dando interpretabilidade.

**Limitações:**
- A base é **desbalanceada** (mais adimplentes que inadimplentes), o que tende a inflar a acurácia — por isso olhamos também precisão, recall e F1.
- A base é antiga (anos 1990, Alemanha); os padrões podem não valer para o contexto atual.
- As funções de pertinência e as faixas de risco (33/66) foram definidas por nós — são **subjetivas**, como ensina a teoria fuzzy.
- Usamos `LabelEncoder` nas variáveis categóricas, o que impõe uma ordem numérica artificial a atributos **nominais** (ex.: finalidade do crédito). Para a Árvore de Decisão o impacto é pequeno, mas o ideal seria *one-hot encoding* nessas colunas.

**Melhorias futuras:**
- Testar outros modelos (Random Forest, Naive Bayes) e comparar.
- Balancear as classes (ex.: SMOTE) e otimizar a profundidade da árvore.
- Refinar as regras fuzzy com um especialista de crédito e expandir a base de regras.


## Conclusão

Desenvolvemos uma solução prática que **integra Machine Learning e Lógica Fuzzy** para análise de risco de crédito. A Árvore de Decisão fornece uma probabilidade de inadimplência aprendida dos dados; o sistema fuzzy a converte, com regras SE–ENTÃO próprias, em um **risco final interpretável** (baixo/médio/alto).

O trabalho exercitou todos os conceitos da disciplina: **fuzzificação, variáveis linguísticas, funções de pertinência, inferência e defuzzificação**, articulados com um modelo supervisionado de classificação e suas **métricas de avaliação**.

---

### Referências
- RUSSELL, S.; NORVIG, P. *Inteligência Artificial*. 3. ed. Rio de Janeiro: Elsevier, 2013.
- ZADEH, L. A. *Fuzzy Sets*. Information and Control, v. 8, n. 3, p. 338–353, 1965.
- Hofmann, H. *Statlog (German Credit Data)*. UCI Machine Learning Repository. Disponível em: https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data
- Scikit-learn — *User Guide*. https://scikit-learn.org/stable/user_guide.html
- Scikit-fuzzy — *Documentation*. https://pythonhosted.org/scikit-fuzzy/
- MALVEZZI, W. *Slides da disciplina de Inteligência Artificial: Lógica Fuzzy e Machine Learning*. Material interno, 2025.
